In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
!{sys.executable} -m pip uninstall torchvision datasets transformers torch scikit-learn -y
!{sys.executable} -m pip install torch transformers datasets scikit-learn -q
!{sys.executable} -m pip install torchvision -q

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: datasets 4.0.0
Uninstalling datasets-4.0.0:
  Successfully uninstalled datasets-4.0.0
Found existing installation: transformers 5.10.2
Uninstalling transformers-5.10.2:
  Successfully uninstalled transformers-5.10.2
Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: scikit-learn 1.6.1
Uninstalling scikit-learn-1.6.1:
  Successfully uninstalled scikit-learn-1.6.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip uninstall torch torchaudio torchvision transformers datasets -y -q
!pip install torch transformers datasets scikit-learn -q
!pip install torchvision torchaudio -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
timm 1.0.27 requires torchvision, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 33.1 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


*   BASE_PATH → Drive'daki veri klasörünün yolu
*   MODEL_NAME → Kullanacağımız BERT modeli (DistilBERT)
*   RANDOM_STATE → Rastgeleliği sabitlemek için (herkes aynı sonucu alır)

*   MAX_LENGTH → Her review'ın tokenize edilirken kesilebileceği maksimum kelime sayısı
*   LABEL2ID / ID2LABEL → Sınıf isimlerini sayıya, sayıları tekrar isme çeviren sözlükler (model sayılarla çalışır)





In [ ]:
BASE_PATH = "/content/drive/MyDrive/amazon-customer-review/data/"
MODEL_NAME = "distilbert-base-uncased"
RANDOM_STATE = 42;
MAX_LENGTH = 128

LABEL2ID = {
    'problem_yok': 0,
    'ürün_kalitesi': 1,
    'ürün_dayanıklılığı': 2,
    'performans': 3,
    'içerik_beklenti': 4
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
df = pd.read_csv(BASE_PATH + "labeled_data.csv")
print(f"Shape: {df.shape}")
print(df['problem_category'].value_counts())

Shape: (116681, 7)
problem_category
problem_yok           106619
ürün_kalitesi           4216
ürün_dayanıklılığı      2525
performans              1857
içerik_beklenti         1464
Name: count, dtype: int64


In [ ]:
print(df.columns.tolist())
print(df.head(2))

['review_headline', 'review_body', 'star_rating', 'verified_purchase', 'helpful_votes', 'total_votes', 'problem_category']
  review_headline                                        review_body  \
0        One Star  Can't finde satelite for DECTVHD Had to get a ...   
1        One Star  I just got them last week. They don't get full...   

   star_rating verified_purchase  helpful_votes  total_votes  \
0            1                 Y              0            0   
1            1                 Y              1            1   

     problem_category  
0  ürün_dayanıklılığı  
1       ürün_kalitesi  


ciddi bir class imbalance var:

problem_yok domine eden çoğunlukta — BERT bunu düzeltmeden sadece problem_yok tahmin eder ve %91 accuracy gösterir ama domine eden class bu olduğu için anlamlı olmaz.

Bu sebeple Undersample + class weight yapacağız

In [ ]:
# Class Imbalance — Her sınıftan max 2000 örnek
dfs = []
for label in df['problem_category'].unique():
    subset = df[df['problem_category'] == label]
    if len(subset) >= 2000:
        subset = subset.sample(n=2000, random_state=RANDOM_STATE)
    dfs.append(subset)

df_balanced = pd.concat(dfs).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
print(df_balanced['problem_category'].value_counts())

problem_category
ürün_dayanıklılığı    2000
ürün_kalitesi         2000
problem_yok           2000
performans            1857
içerik_beklenti       1464
Name: count, dtype: int64


In [ ]:
# Label Encoding
df_balanced['label'] = df_balanced['problem_category'].map(LABEL2ID)

# Train / Val / Test Split
train_df, test_df = train_test_split(df_balanced, test_size=0.2, stratify=df_balanced['label'], random_state=RANDOM_STATE)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df['label'], random_state=RANDOM_STATE)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 6710, Val: 746, Test: 1865


In [ ]:
# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# HuggingFace Dataset formatına çevir
def tokenize(batch):
    return tokenizer(batch['review_body'], truncation=True, padding='max_length', max_length=MAX_LENGTH)

train_dataset = Dataset.from_pandas(train_df[['review_body', 'label']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['review_body', 'label']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['review_body', 'label']].reset_index(drop=True))

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization tamamlandı.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/6710 [00:00<?, ? examples/s]

Map:   0%|          | 0/746 [00:00<?, ? examples/s]

Map:   0%|          | 0/1865 [00:00<?, ? examples/s]

Tokenization tamamlandı.


In [ ]:
# Model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Training Arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/amazon-customer-review/model",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_dir="./logs",
    logging_steps=50,
    warmup_steps=100,
    weight_decay=0.01,
    fp16=True,
)

print("Model ve training arguments hazır.")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model ve training arguments hazır.


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average='weighted')
    acc = (preds == labels).mean()
    return {'accuracy': acc, 'f1': f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Trainer hazır.")

Trainer hazır.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.482638,0.505811,0.828418,0.828171
2,0.321755,0.494565,0.843164,0.842486
3,0.119997,0.541370,0.855228,0.855372


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1260, training_loss=0.40959968912223027, metrics={'train_runtime': 218.0115, 'train_samples_per_second': 92.335, 'train_steps_per_second': 5.78, 'total_flos': 666677849587200.0, 'train_loss': 0.40959968912223027, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate(test_dataset)
print(eval_results)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.119997,0.529814,3,0.838070,0.836562


{'eval_loss': 0.5298137664794922, 'eval_accuracy': 0.8380697050938338, 'eval_f1': 0.8365615366540685}


In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)

print(classification_report(predictions.label_ids, preds, target_names=list(LABEL2ID.keys())))

                    precision    recall  f1-score   support

       problem_yok       0.89      0.85      0.87       400
     ürün_kalitesi       0.80      0.82      0.81       400
ürün_dayanıklılığı       0.85      0.71      0.77       400
        performans       0.80      0.93      0.86       372
   içerik_beklenti       0.86      0.90      0.88       293

          accuracy                           0.84      1865
         macro avg       0.84      0.84      0.84      1865
      weighted avg       0.84      0.84      0.84      1865



In [ ]:
model.save_pretrained("/content/drive/MyDrive/amazon-customer-review/model/bert_final")
tokenizer.save_pretrained("/content/drive/MyDrive/amazon-customer-review/model/bert_final")
print("Model kaydedildi.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model kaydedildi.


In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="/content/drive/MyDrive/amazon-customer-review/model/bert_final",
    tokenizer="/content/drive/MyDrive/amazon-customer-review/model/bert_final"
)

test_reviews = [
    "Good but not working",
    "Great product, very fast delivery",
    "Broke after one week, very disappointed",
]

for review in test_reviews:
    result = classifier(review)
    print(f"Review: {review}")
    print(f"Tahmin: {result[0]['label']} (skor: {result[0]['score']:.2f})")
    print()

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Review: Good but not working
Tahmin: ürün_dayanıklılığı (skor: 0.99)

Review: Great product, very fast delivery
Tahmin: problem_yok (skor: 0.99)

Review: Broke after one week, very disappointed
Tahmin: ürün_kalitesi (skor: 0.91)



In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "/content/drive/MyDrive/amazon-customer-review/model/bert_final"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.push_to_hub("arcasoyece/amazon-review-classifier")
tokenizer.push_to_hub("arcasoyece/amazon-review-classifier")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ojrb0q3/model.safetensors:  12%|#1        | 32.0MB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/arcasoyece/amazon-review-classifier/commit/29f124b89adc7d930d7c5698770ac5ad45a0b00d', commit_message='Upload tokenizer', commit_description='', oid='29f124b89adc7d930d7c5698770ac5ad45a0b00d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/arcasoyece/amazon-review-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='arcasoyece/amazon-review-classifier'), pr_revision=None, pr_num=None)

In [ ]:
from transformers import pipeline
classifier = pipeline("text-classification", model="arcasoyece/amazon-review-classifier")

config.json:   0%|          | 0.00/1.04k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

In [ ]:
LABEL2ID = {
    'problem_yok': 0,
    'ürün_kalitesi': 1,
    'ürün_dayanıklılığı': 2,
    'teknik_sorun': 3,
    'içerik_beklenti': 4,
    'kargo_teslimat': 5,
    'satıcı': 6
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

In [ ]:
df = pd.read_csv(BASE_PATH + "labeled_data_full.csv")

---
# Bölüm 2: BERT v2 — 7 Kategori, Class-Weighted, 533k Veri

İlk eğitimde (yukarıda) 5 kategori + undersampling kullanıldı (Test Macro F1: ~0.84, 6710 train örneği).

Bu bölümde:
- Relabeling pipeline'ı ile eklenen 2 yeni kategori: `kargo_teslimat`, `satıcı`
- Tüm 533.680 satır kullanıldı (undersample yok)
- Class weighting (`compute_class_weight`) ile sınıf dengesizliği çözüldü
- 7 kategori: problem_yok, ürün_kalitesi, ürün_dayanıklılığı, teknik_sorun, içerik_beklenti, kargo_teslimat, satıcı

kullanılarak model sıfırdan (`distilbert-base-uncased`) yeniden eğitildi.

In [ ]:
df = pd.read_csv(BASE_PATH + "labeled_data_full.csv")
print(df.shape)
print(df['problem_category'].value_counts())

(533680, 7)
problem_category
problem_yok           376495
teknik_sorun           87295
ürün_kalitesi          56938
kargo_teslimat          6303
ürün_dayanıklılığı      3120
satıcı                  2883
içerik_beklenti          646
Name: count, dtype: int64


In [ ]:
df['label'] = df['problem_category'].map(LABEL2ID)

train_df, test_df = train_test_split(df, test_size=0.1, stratify=df['label'], random_state=RANDOM_STATE)
train_df, val_df = train_test_split(train_df, test_size=0.05, stratify=train_df['label'], random_state=RANDOM_STATE)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 456296, Val: 24016, Test: 53368


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    'balanced',
    classes=np.array(list(LABEL2ID.values())),
    y=train_df['label'].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

[  0.20249996   1.33899887  24.43221247   0.87335061 118.08902692
  12.09596268  26.44427702]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['review_body'], truncation=True, padding='max_length', max_length=MAX_LENGTH)

train_dataset = Dataset.from_pandas(train_df[['review_body', 'label']].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[['review_body', 'label']].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[['review_body', 'label']].reset_index(drop=True))

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

print("Tokenization tamamlandı.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/456296 [00:00<?, ? examples/s]

Map:   0%|          | 0/24016 [00:00<?, ? examples/s]

Map:   0%|          | 0/53368 [00:00<?, ? examples/s]

Tokenization tamamlandı.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor.to(logits.device))
        loss = loss_fct(logits.view(-1, len(LABEL2ID)), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(
    output_dir=BASE_PATH + "model/bert_v2",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=200,
    warmup_steps=500,
    weight_decay=0.01,
    fp16=True,
)

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average='macro')
    acc = (preds == labels).mean()
    return {'accuracy': acc, 'f1': f1}

In [ ]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)
print("Trainer hazır.")

Trainer hazır.


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.388155,0.335597,0.858553,0.759861
2,0.299899,0.304867,0.903398,0.811477
3,0.156129,0.409709,0.918013,0.857108


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=42780, training_loss=0.3593861364995057, metrics={'train_runtime': 3982.3922, 'train_samples_per_second': 343.735, 'train_steps_per_second': 10.742, 'total_flos': 4.533730037436211e+16, 'train_loss': 0.3593861364995057, 'epoch': 3.0})

In [ ]:
eval_results = trainer.evaluate(test_dataset)
print(eval_results)

predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
print(classification_report(predictions.label_ids, preds, target_names=list(LABEL2ID.keys())))

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.156129,0.490593,3,0.916823,0.837798


{'eval_loss': 0.49059250950813293, 'eval_accuracy': 0.9168228151701394, 'eval_f1': 0.8377976363922975}


                    precision    recall  f1-score   support

       problem_yok       0.96      0.93      0.95     37650
     ürün_kalitesi       0.84      0.89      0.86      5694
ürün_dayanıklılığı       0.84      0.94      0.88       312
      teknik_sorun       0.81      0.88      0.84      8729
   içerik_beklenti       0.70      0.91      0.79        65
    kargo_teslimat       0.70      0.78      0.74       630
            satıcı       0.77      0.82      0.79       288

          accuracy                           0.92     53368
         macro avg       0.80      0.88      0.84     53368
      weighted avg       0.92      0.92      0.92     53368



## Sonuçlar (BERT v2 — 7 Kategori, Class-Weighted, 533k Veri)

| Kategori | Precision | Recall | F1 | Support |
|----------|-----------|--------|-----|---------|
| problem_yok | 0.96 | 0.93 | 0.95 | 37650 |
| ürün_kalitesi | 0.84 | 0.89 | 0.86 | 5694 |
| ürün_dayanıklılığı | 0.84 | 0.94 | 0.88 | 312 |
| teknik_sorun | 0.81 | 0.88 | 0.84 | 8729 |
| içerik_beklenti | 0.70 | 0.91 | 0.79 | 65 |
| kargo_teslimat | 0.70 | 0.78 | 0.74 | 630 |
| satıcı | 0.77 | 0.82 | 0.79 | 288 |

**Accuracy: 0.92** · **Macro F1: 0.84** · **Weighted F1: 0.92**

### v1 vs v2 Karşılaştırması

| | v1 (5 kategori, undersample) | v2 (7 kategori, class-weighted, 533k) |
|---|---|---|
| Eğitim verisi | 6.710 | 456.296 |
| Macro F1 | 0.84 | 0.84 |
| Accuracy | 0.84 | 0.92 |
| Kategori sayısı | 5 | 7 |

v2, daha fazla kategori ve gerçek veri dağılımıyla (undersample yok) eğitilmesine rağmen
benzer macro F1 ile **daha yüksek accuracy** sağladı. Ayrıca 2 yeni kategori
(`kargo_teslimat`, `satıcı`) az veriyle bile 0.74+ F1 ile öğrenildi.
Model2 eski modelle değiştirildi.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoModelForSequenceClassification, AutoTokenizer

BASE_PATH = "/content/drive/MyDrive/amazon-customer-review/data/"

# Drive'da kaydedilmiş checkpoint'i bul
import os
checkpoint_dir = BASE_PATH + "model/bert_v2"
checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint')]
print(checkpoints)

Mounted at /content/drive
['checkpoint-14260', 'checkpoint-28520', 'checkpoint-42780']


In [5]:
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get('HF_TOKEN'))

model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir + "/checkpoint-42780")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_dir + "/checkpoint-42780")

model.push_to_hub("arcasoyece/amazon-review-classifier")
tokenizer.push_to_hub("arcasoyece/amazon-review-classifier")
print("v2 modeli HuggingFace'e yüklendi.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ..._7u0el8/model.safetensors:   0%|          |  575kB /  268MB            

v2 modeli HuggingFace'e yüklendi.
